
| <h1> **Hands-on Activity 8.1** </h1> | <h1> **Aggregating Pandas DataFrames** </h1> |
|--- | --- |
Name: | Rubang, Jethro Aaron S.<br>
Course and Section: |  CPE 311-CPE22S3<br>
Instructor: | Eng. Neal Barton James Matira
Date Performed: | March 02, 2026
Date Submitted: | March 02, 2026


<hr>


## 8.1.1 Intended Learning Outcomes
After this activity, the student should be able to:
- Demonstrate querying and merging of dataframes
- Perform advanced calculations on dataframes
- Aggregate dataframes with pandas and numpy
- Work with time series data

## 8.1.2 Resources
- Computing Environment using Python 3.x
- Attached Datasets (under Instructional Materials)


## 8.1.3 Procedures
The procedures can be found in the canvas module. Check the following under topics:
- 8.1 Weather Data Collection
- 8.2 Querying and Merging
- 8.3 Dataframe Operations
- 8.4 Aggregations
- 8.5 Time Series


### 8.1 Weather Data Collection
#### Collecting weather data from an API
##### About the data
In this notebook, we will be collecting daily weather data from the National Centers for Environmental Information (NCEI) API. We will use the Global Historical Climatology
Network - Daily (GHCND) data set; see the documentation here.

Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the
NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for the NCEI weather API to find the updated one.

#### Using the NCEI API
Paste your token below.

In [3]:
import requests
def make_request(endpoint, payload=None):

 return requests.get(
 f'https://www.ncdc.noaa.gov/cdo-web/api/v2/{endpoint}',
 headers={
 'token': 'EgJhuSMiPMBRMwkoUKTNbynhvWMNgLzn'
 },
 params=payload
 )

#### Collect All Data Points for 2018 In NYC (Various Stations)
We can make a loop to query for all the data points one day at a time. Here we create a list of all the results:

In [4]:
import datetime
from IPython import display 
current = datetime.date(2018, 1, 1)
end = datetime.date(2019, 1, 1)
results = []
while current < end:

 display.clear_output(wait=True)
 display.display(f'Gathering data for {str(current)}')
 
 response = make_request(
 'data',
 {
 'datasetid' : 'GHCND', # Global Historical Climatology Network - Daily (GHCND) dataset
 'locationid' : 'CITY:US360019', # NYC
 'startdate' : current,
 'enddate' : current,
 'units' : 'metric',
 'limit' : 1000 # max allowed
 }
 )
 if response.ok:
 # we extend the list instead of appending to avoid getting a nested list
    results.extend(response.json()['results'])
 # update the current date to avoid an infinite loop
 current += datetime.timedelta(days=1)


'Gathering data for 2018-12-31'

Now, we can create a dataframe with all this data. Notice there are multiple stations with values for each datatype on a given day. We don't know what the stations are, but we
can look them up and add them to the data:

In [5]:
import pandas as pd
df = pd.DataFrame(results)
df.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


Save this data to a file:

In [6]:
df.to_csv('nyc_weather.csv', index=False)

and write it to the database:

In [7]:
import sqlite3
with sqlite3.connect('weather.db') as connection:
 df.to_sql(
 'weather', connection, index=False, if_exists='replace'
 )

For learning about merging dataframes, we will also get the data mapping station IDs to information about the station:

In [8]:
response = make_request(
 'stations',
 {
 'datasetid' : 'GHCND', # Global Historical Climatology Network - Daily (GHCND) dataset
 'locationid' : 'CITY:US360019', # NYC
 'limit' : 1000 # max allowed
 }
)
stations = pd.DataFrame(response.json()['results'])[['id', 'name', 'latitude', 'longitude', 'elevation']]
stations.to_csv('weather_stations.csv', index=False)
with sqlite3.connect('weather.db') as connection:
 stations.to_sql(
 'stations', connection, index=False, if_exists='replace'
    )

### 8.2 Querying and Merging
#### Database-style Operations on Dataframes
##### About the data
In this notebook, we will be collecting daily weather data from the National Centers for Environmental Information (NCEI) API. We will use the Global Historical Climatology
Network - Daily (GHCND) data set; see the documentation here.

Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the
NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for the NCEI weather API to find the updated one.

#### Background on the data
Data meanings:
- PRCP : precipitation in millimeters
- SNOW : snowfall in millimeters
- SNWD : snow depth in millimeters
- TMAX : maximum daily temperature in Celsius
- TMIN : minimum daily temperature in Celsius
- TOBS : temperature at time of observation in Celsius
- WESF : water equivalent of snow in millimeters

#### Setup

In [9]:
import pandas as pd
weather = pd.read_csv('nyc_weather.csv')
weather.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


#### Querying DataFrames
The query() method is an easier way of filtering based on some criteria. For example, we can use it to find all entries where snow was recorded:

In [10]:
snow_data = weather.query('datatype == "SNOW" and value > 0')
snow_data.head()

,date,datatype,station,attributes,value
127,2018-01-01T00:00:00,SNOW,GHCND:US1NYWC0019,",,N,1700",25.0
816,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1600",229.0
819,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0830",10.0
823,2018-01-04T00:00:00,SNOW,GHCND:US1NJBG0018,",,N,0910",46.0
830,2018-01-04T00:00:00,SNOW,GHCND:US1NJES0018,",,N,0700",10.0


This is equivalent to quering the data/weather.db SQLite database for SELECT * FROM weather WHERE datatype == "SNOW" AND value > 0 :

In [11]:
import sqlite3
with sqlite3.connect('weather.db') as connection:
    snow_data_from_db = pd.read_sql(
'SELECT * FROM weather WHERE datatype == "SNOW" AND value > 0',
    connection
    )
snow_data.reset_index().drop(columns='index').equals(snow_data_from_db)

True

Note this is also equivalent to creating Boolean masks:

In [12]:
weather[(weather.datatype == 'SNOW') & (weather.value > 0)].equals(snow_data)

True

#### Merging DataFrames
We have data for many different stations each day; however, we don't know what the stations are just their IDs. We can join the data in the data/weather_stations.csv
file which contains information from the stations endpoint of the NCEI API. Consult the weather_data_collection.ipynb notebook to see how this was collected. It looks like this:

In [13]:
station_info = pd.read_csv('weather_stations.csv')
station_info.head()

,id,name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


As a reminder, the weather data looks like this:

In [14]:
weather.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


We can join our data by matching up the station_info.id column with the weather.station column. Before doing that though, let's see how many unique values we have:

In [15]:
station_info.id.describe()

count                   330
unique                  330
top       GHCND:USW00094789
freq                      1
Name: id, dtype: object

While station_info has one row per station, the weather dataframe has many entries per station. Notice it also has fewer uniques:

In [16]:
weather.station.describe()

count                 90788
unique                  114
top       GHCND:USW00014734
freq                   6705
Name: station, dtype: object

When working with joins, it is important to keep an eye on the row count. Some join types will lead to data loss:

In [17]:
station_info.shape[0], weather.shape[0]

(330, 90788)

Since we will be doing this often, it makes more sense to write a function:

In [18]:
def get_row_count(*dfs):
    return [df.shape[0] for df in dfs]
get_row_count(station_info, weather)

[330, 90788]

The map() function is more efficient than list comprehensions. We can couple this with getattr() to grab any attribute for multiple dataframes:

In [19]:
def get_info(attr, *dfs):
    return list(map(lambda x: getattr(x, attr), dfs))
get_info('shape', station_info, weather)

[(330, 5), (90788, 5)]

By default merge() performs an inner join. We simply specify the columns to use for the join. The left dataframe is the one we call merge() on, and the right one is passed in as an argument:

In [20]:
inner_join = weather.merge(station_info, left_on='station', right_on='id')
inner_join.sample(5, random_state=0)

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
62445,2018-09-08T00:00:00,SNOW,GHCND:US1NJPS0012,",,N,0700",0.0,GHCND:US1NJPS0012,"LITTLE FALLS TWP 0.5 WNW, NJ US",40.879633,-74.227041,58.5
84506,2018-12-06T00:00:00,WDF5,GHCND:USW00094745,",,W,",280.0,GHCND:USW00094745,"WESTCHESTER CO AIRPORT, NY US",41.062360,-73.704540,112.9
48946,2018-07-14T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,0700",0.0,GHCND:US1NJBG0015,"NORTH ARLINGTON 0.7 WNW, NJ US",40.791492,-74.139790,17.7
68072,2018-10-01T00:00:00,SNOW,GHCND:US1NJPS0012,",,N,0700",0.0,GHCND:US1NJPS0012,"LITTLE FALLS TWP 0.5 WNW, NJ US",40.879633,-74.227041,58.5
74974,2018-10-29T00:00:00,SNWD,GHCND:USC00301309,",,7,0700",0.0,GHCND:USC00301309,"CENTERPORT, NY US",40.883450,-73.373090,9.1


We can remove the duplication of information in the station and id columns by renaming one of them before the merge and then simply using on :

In [21]:
weather.merge(station_info.rename(dict(id='station'), axis=1), on='station').sample(5, random_state=0)

,date,datatype,station,attributes,value,name,latitude,longitude,elevation
62445,2018-09-08T00:00:00,SNOW,GHCND:US1NJPS0012,",,N,0700",0.0,"LITTLE FALLS TWP 0.5 WNW, NJ US",40.879633,-74.227041,58.5
84506,2018-12-06T00:00:00,WDF5,GHCND:USW00094745,",,W,",280.0,"WESTCHESTER CO AIRPORT, NY US",41.062360,-73.704540,112.9
48946,2018-07-14T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,0700",0.0,"NORTH ARLINGTON 0.7 WNW, NJ US",40.791492,-74.139790,17.7
68072,2018-10-01T00:00:00,SNOW,GHCND:US1NJPS0012,",,N,0700",0.0,"LITTLE FALLS TWP 0.5 WNW, NJ US",40.879633,-74.227041,58.5
74974,2018-10-29T00:00:00,SNWD,GHCND:USC00301309,",,7,0700",0.0,"CENTERPORT, NY US",40.883450,-73.373090,9.1


We are losing stations that don't have weather observations associated with them, if we don't want to lose these rows, we perform a right or left join instead of the inner join:

In [22]:
left_join = station_info.merge(weather, left_on='id', right_on='station', how='left')
right_join = weather.merge(station_info, left_on='station', right_on='id', how='right')
right_join.tail()

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
90999,2018-12-31T00:00:00,WDF5,GHCND:USW00094789,",,W,",130.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
91000,2018-12-31T00:00:00,WSF2,GHCND:USW00094789,",,W,",9.8,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
91001,2018-12-31T00:00:00,WSF5,GHCND:USW00094789,",,W,",12.5,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
91002,2018-12-31T00:00:00,WT01,GHCND:USW00094789,",,W,",1.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7
91003,2018-12-31T00:00:00,WT02,GHCND:USW00094789,",,W,",1.0,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.63915,-73.7639,2.7


The left and right join as we performed above are equivalent because the side that we kept the rows without matches was the same in both cases:

In [23]:
left_join.sort_index(axis=1).sort_values(['date', 'station']).reset_index().drop(columns='index').equals(right_join.sort_index(axis=1).sort_values(['date', 'station']).reset_index().drop(columns='index'))

True

Note we have additional rows in the left and right joins because we kept all the stations that didn't have weather observations:

In [24]:
get_info('shape', inner_join, left_join, right_join)

[(90788, 10), (91004, 10), (91004, 10)]

If we query the station information for stations that have NY in their name, believing that to be all the stations that record weather data for NYC and perform an outer join, we can see where the mismatches occur:

In [25]:
outer_join = weather.merge(station_info[station_info.name.str.contains('NY')],left_on='station', right_on='id', how='outer', indicator=True)
pd.concat([
    outer_join.sample(4, random_state=0), 
    outer_join[outer_join.station.isna()].head(2)
])


,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation,_merge
45898,2018-09-14T00:00:00,TMAX,GHCND:USC00284987,",,7,0630",22.8,NaN,NaN,NaN,NaN,NaN,left_only
27559,2018-01-31T00:00:00,PRCP,GHCND:US1NYNS0018,"T,,N,0630",0.0,GHCND:US1NYNS0018,"HICKSVILLE 1.3 ENE, NY US",40.768654,-73.501701,45.7,both
64802,2018-12-09T00:00:00,TMIN,GHCND:USW00014734,",,W,2400",-3.8,NaN,NaN,NaN,NaN,NaN,left_only
64823,2018-12-10T00:00:00,WSF2,GHCND:USW00014734,",,W,",7.2,NaN,NaN,NaN,NaN,NaN,left_only
7104,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJHD0018,"KEARNY 1.7 NNW, NJ US",40.774342,-74.137109,25.6,right_only
15070,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJMS0036,"PARSIPPANY TROY HILLS TWP 2.1 E, NJ US",40.865600,-74.385100,64.3,right_only


These joins are equivalent to their SQL counterparts. Below is the inner join. Note that to use equals() you will have to do some manipulation of the dataframes to line them up:

In [26]:
import sqlite3
with sqlite3.connect('weather.db') as connection:
    inner_join_from_db = pd.read_sql(
'SELECT * FROM weather JOIN stations ON weather.station == stations.id',
    connection
    )
inner_join_from_db.shape == inner_join.shape

True

Revisit the dirty data from the previous module.

In [28]:
dirty_data = pd.read_csv('nyc_weather.csv', index_col='date').drop_duplicates()
dirty_data = dirty_data[dirty_data.datatype != 'SNWD']
dirty_data.head()

,datatype,station,attributes,value
date,,,,
2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,0800",0.0
2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,1050",0.0
2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,1050",0.0
2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,0920",0.0
2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,0920",0.0


We need to create two dataframes for the join. We will drop some unecessary columns as well for easier viewing:

## 8.1.4 Data Analysis
### 8.1 Weather Data Collection
I learned that there are more ways of getting data, I have learned webscraping and using straight CSVs before. In this activity, we used API requests and using sqlite3 to transform this into usable data. So far, I can see that using CSVs is the way if the data is readily available, using API requests is good if a company or data source allows it, and webscraping if the other two options are not available.

## 8.1.5 Supplementary Activity
Using the CSV files provided and what we have learned so far in this module complete the following exercises:
1. With the earthquakes.csv file, select all the earthquakes in Japan with a magType of mb and a magnitude of 4.9 or greater.
2. Create bins for each full number of magnitude (for example, the first bin is 0-1, the second is 1-2, and so on) with a magType of ml and count how many are in each bin.
3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:
- Mean of the opening price
- Maximum of the high price
- Minimum of the low price
- Mean of the closing price
- Sum of the volume traded
4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum
magnitude that was observed for each combination. Put the magType along the columns.
5. Calculate the rolling 60-day aggregations of OHLC data by ticker for the FAANG data. Use the same aggregations as exercise no. 3.
6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.
7. Calculate the Z-scores for each numeric column of Netflix's data (ticker is NFLX) using apply().
8. Add event descriptions:
- Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values:
  - ticker: 'FB'
  - date: ['2018-07-25', '2018-03-19', '2018-03-20']
  - event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
- Set the index to ['date', 'ticker']
- Merge this data with the FAANG data using an outer join
9. Use the transform() method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values
for the first date in the data for that ticker. This is referred to as an index, and the data for the first date is the base (https://e​c.e​uropa.e​u/e​urostat/s​tatistics-e​xplained/​
index.p​hp/ Beginners:Statisticalc​oncept-​I​ndexa​ndb​asey​ear). When data is in this format, we can easily see growth over time. Hint: transform() can take a function name.


In [29]:
#1. With the earthquakes.csv file, select all the earthquakes in Japan with a magType of mb and a magnitude of 4.9 or greater.
earthquakes = pd.read_csv('earthquakes.csv')
japan_earthquakes = earthquakes[earthquakes.parsed_place == 'Japan']
japanmb_earthquakes = japan_earthquakes[
    (japan_earthquakes.magType == 'mb') & (japan_earthquakes.mag >= 4.9)
]
japanmb_earthquakes.head()

,mag,magType,time,place,tsunami,parsed_place
1563,4.9,mb,1538977532250,"293km ESE of Iwo Jima, Japan",0,Japan
2576,5.4,mb,1538697528010,"37km E of Tomakomai, Japan",0,Japan
3072,4.9,mb,1538579732490,"15km ENE of Hasaki, Japan",0,Japan
3632,4.9,mb,1538450871260,"53km ESE of Hitachi, Japan",0,Japan


In [30]:
#2. Create bins for each full number of magnitude (for example, the first bin is 0-1, the second is 1-2, and so on) with a magType of ml and count how many are in each bin.
ml_earthquakes = earthquakes[earthquakes.magType == 'ml']
ml_earthquakes['magnitude_bin'] = ml_earthquakes['mag'].apply(lambda x: int(x))
magnitude_counts = ml_earthquakes.groupby('magnitude_bin').size()
magnitude_counts

C:\Users\jethr\AppData\Local\Temp\ipykernel_24976\2807307293.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_earthquakes['magnitude_bin'] = ml_earthquakes['mag'].apply(lambda x: int(x))


magnitude_bin
-1      13
 0    2518
 1    3126
 2     985
 3     153
 4       6
 5       2
dtype: int64

In [31]:
#3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:
faang = pd.read_csv('faang.csv')
faang['date'] = pd.to_datetime(faang['date'])
faang.set_index('date', inplace=True)
monthly_data = faang.groupby('ticker').resample('ME').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})
print(monthly_data)

                        open       high        low      close     volume
ticker date                                                             
AAPL   2018-01-31   166.9271   176.6782   161.5708   164.2490  659679440
       2018-02-28   163.9890   177.9059   147.9865   175.4484  927894473
       2018-03-31   175.8621   180.7477   162.4660   165.2635  713727447
       2018-04-30   165.3620   176.2526   158.2207   162.7812  666360147
       2018-05-31   163.9142   187.9311   162.7911   184.7768  620976206
       2018-06-30   185.8855   192.0247   178.7056   183.0366  527624365
       2018-07-31   181.7610   193.7650   181.3655   188.1585  393843881
       2018-08-31   196.8995   227.1001   195.0999   225.8697  700318837
       2018-09-30   226.6437   227.8939   213.6351   223.9943  678972040
       2018-10-31   226.1872   231.6645   204.4963   217.1675  789748068
       2018-11-30   217.3561   220.6405   169.5328   177.8173  961321947
       2018-12-31   183.6722   184.1501   145.9639 

In [32]:
#4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum
crosstab = pd.crosstab(earthquakes.tsunami, earthquakes.magType, values=earthquakes.mag, aggfunc='max')
print(crosstab)

magType   mb  mb_lg    md   mh   ml  ms_20    mw  mwb  mwr  mww
tsunami                                                        
0        5.6    3.5  4.11  1.1  4.2    NaN  3.83  5.8  4.8  6.0
1        6.1    NaN   NaN  NaN  5.1    5.7  4.41  NaN  NaN  7.5


In [33]:
#5. Calculate the rolling 60-day aggregations of OHLC data by ticker for the FAANG data. Use the same aggregations as exercise no. 3.
rolling_data = faang.groupby('ticker').rolling(window=60).agg({
    'open': lambda x: x.iloc[0],
    'high': 'max',
    'low': 'min',
    'close': lambda x: x.iloc[-1],
    'volume': 'sum'
}).reset_index()
print(rolling_data)

     ticker       date    open      high     low    close       volume
0      AAPL 2018-01-02     NaN       NaN     NaN      NaN          NaN
1      AAPL 2018-01-03     NaN       NaN     NaN      NaN          NaN
2      AAPL 2018-01-04     NaN       NaN     NaN      NaN          NaN
3      AAPL 2018-01-05     NaN       NaN     NaN      NaN          NaN
4      AAPL 2018-01-08     NaN       NaN     NaN      NaN          NaN
...     ...        ...     ...       ...     ...      ...          ...
1250   NFLX 2018-12-24  379.24  386.7999  233.68  233.880  811001766.0
1251   NFLX 2018-12-26  375.85  386.7999  231.23  253.670  818289623.0
1252   NFLX 2018-12-27  384.38  386.7999  231.23  255.565  822148280.0
1253   NFLX 2018-12-28  378.53  380.9300  231.23  256.080  824496849.0
1254   NFLX 2018-12-31  375.88  380.0000  231.23  267.660  832207164.0

[1255 rows x 7 columns]


In [36]:
#6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.
pivot_table = faang.pivot_table(index='ticker', values=['open', 'high', 'low', 'close', 'volume'], aggfunc='mean')
print(pivot_table)

              close         high          low         open        volume
ticker                                                                  
AAPL     186.986218   188.906858   185.135729   187.038674  3.402145e+07
AMZN    1641.726175  1662.839801  1619.840398  1644.072669  5.649563e+06
FB       171.510936   173.615298   169.303110   171.454424  2.768798e+07
GOOG    1113.225139  1125.777649  1101.001594  1113.554104  1.742645e+06
NFLX     319.290299   325.224583   313.187273   319.620533  1.147030e+07


In [37]:
#7. Calculate the Z-scores for each numeric column of Netflix's data (ticker is NFLX) using apply().
netflix_data = faang[faang['ticker'] == 'NFLX']
z_scores = netflix_data[['open', 'high', 'low', 'close', 'volume']].apply(lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0)
print(z_scores)

                open      high       low     close    volume
date                                                        
2018-01-02 -2.500753 -2.516023 -2.410226 -2.416644 -0.088760
2018-01-03 -2.380291 -2.423180 -2.285793 -2.335286 -0.507606
2018-01-04 -2.296272 -2.406077 -2.234616 -2.323429 -0.959287
2018-01-05 -2.275014 -2.345607 -2.202087 -2.234303 -0.782331
2018-01-08 -2.218934 -2.295113 -2.143759 -2.192192 -1.038531
...              ...       ...       ...       ...       ...
2018-12-24 -1.571478 -1.518366 -1.627197 -1.745946 -0.339003
2018-12-26 -1.735063 -1.439978 -1.677339 -1.341402  0.517040
2018-12-27 -1.407286 -1.417785 -1.495805 -1.302664  0.134868
2018-12-28 -1.248762 -1.289018 -1.297285 -1.292137 -0.085164
2018-12-31 -1.203817 -1.122354 -1.088531 -1.055420  0.359444

[251 rows x 5 columns]


In [38]:
#8. Add event descriptions:
event_df = pd.DataFrame({
    'ticker': ['FB', 'FB', 'FB'],
    'date': ['2018-07-25', '2018-03-19', '2018-03-20'],
    'event': ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
})
event_df['date'] = pd.to_datetime(event_df['date'])
faang_reset = faang.reset_index()
faang_with_events = faang_reset.merge(event_df, on=['date', 'ticker'], how='outer')
faang_with_events.set_index('date', inplace=True)
print(faang_with_events)

           ticker       open       high        low      close    volume event
date                                                                         
2018-01-02   AAPL   166.9271   169.0264   166.0442   168.9872  25555934   NaN
2018-01-02   AMZN  1172.0000  1190.0000  1170.5100  1189.0100   2694494   NaN
2018-01-02     FB   177.6800   181.5800   177.5500   181.4200  18151903   NaN
2018-01-02   GOOG  1048.3400  1066.9400  1045.2300  1065.0000   1237564   NaN
2018-01-02   NFLX   196.1000   201.6500   195.4200   201.0700  10966889   NaN
...           ...        ...        ...        ...        ...       ...   ...
2018-12-31   AAPL   157.8529   158.6794   155.8117   157.0663  35003466   NaN
2018-12-31   AMZN  1510.8000  1520.7600  1487.0000  1501.9700   6954507   NaN
2018-12-31     FB   134.4500   134.6400   129.9500   131.0900  24625308   NaN
2018-12-31   GOOG  1050.9600  1052.7000  1023.5900  1035.6100   1493722   NaN
2018-12-31   NFLX   260.1600   270.1001   260.0000   267.6600  1

In [39]:
#9. Use the transform() method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values
faang_indexed = faang.groupby('ticker').transform(lambda x: x / x.iloc[0])
print(faang_indexed)

                open      high       low     close    volume
date                                                        
2018-01-02  1.000000  1.000000  1.000000  1.000000  1.000000
2018-01-03  1.023638  1.017623  1.021290  1.017914  0.930292
2018-01-04  1.040635  1.025498  1.036889  1.016040  0.764707
2018-01-05  1.044518  1.029298  1.041566  1.029931  0.747830
2018-01-08  1.053579  1.040313  1.049451  1.037813  0.991341
...              ...       ...       ...       ...       ...
2018-12-24  0.928993  0.940578  0.928131  0.916638  1.285047
2018-12-26  0.943406  0.974750  0.940463  0.976019  1.917695
2018-12-27  0.970248  0.978396  0.953857  0.980169  1.704782
2018-12-28  1.001221  0.989334  0.988395  0.973784  1.142383
2018-12-31  1.002499  0.986653  0.979296  0.972404  1.206986

[1255 rows x 5 columns]


## Conclusion

I learned the importance of aggregating dataframes using pandas and numpy. This skill is essential for summarizing large datasets and extracting meaningful statistics that are often hidden in raw data. I was also able to work with time series data, understanding how to manipulate and analyze information over specific intervals. Overall, doing these procedures and supplementary activities has greatly enhanced my ability to handle data-driven tasks, and I am look forward to using these techniques to more complex data science projects and activities in the future.